# Data Load

In [ ]:
import pandas as pd

dnv = pd.read_csv("/path/to/data/variant.csv")

# Variant Pooling

## 1. DNABERT

In [3]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, BertConfig


def seq2kmer_list(seq: str, k: int):
    return [seq[x:x + k] for x in range(len(seq) + 1 - k)]


class DNABERT_MutPool:

    def __init__(
        self,
        model_path: str,
        kmer: int,
        device: str = "cuda:0",
    ):

        self.device = torch.device(device)
        self.k = int(kmer)

        self.config = BertConfig.from_pretrained(model_path)
        self.config.output_hidden_states = True
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(
            model_path,
            trust_remote_code=True,
            config=self.config,
        )
        self.model.to(self.device)
        self.model.eval()

    def get_mut_emb_raw(
        self,
        seq_df: pd.DataFrame,
        seq_col: str,
        mut_idx_col: str,
    ) -> pd.DataFrame:

        rows = []
        max_kmer_tokens = 510

        with torch.no_grad():
            for _, row in tqdm(seq_df.iterrows(), total=len(seq_df)):
                sample = row["vcf_iid"]
                varpos = row["variant"]
                seq = row[seq_col]
                mut_idx_list = row[mut_idx_col]

                if mut_idx_list is None or len(mut_idx_list) == 0:
                    continue

                mut_idx_set = set(mut_idx_list)

                kmers = seq2kmer_list(seq, self.k)
                n_kmers = len(kmers)
                if n_kmers <= 0:
                    continue

                mut_token_ids_full = []
                for i_tok in range(n_kmers):
                    tok_start = i_tok
                    tok_end = i_tok + self.k
                    covered = range(tok_start, tok_end)

                    if any(pos in mut_idx_set for pos in covered):
                        mut_token_ids_full.append(i_tok)

                if len(mut_token_ids_full) == 0:
                    continue

                t_min = min(mut_token_ids_full)
                t_max = max(mut_token_ids_full)
                span = t_max - t_min + 1

                if span >= max_kmer_tokens:
                    win_start = t_min
                    win_end = min(t_min + max_kmer_tokens, n_kmers)
                else:
                    extra = max_kmer_tokens - span
                    left_margin = extra // 2
                    right_margin = extra - left_margin

                    win_start = max(0, t_min - left_margin)
                    win_end = min(n_kmers, t_max + right_margin + 1)

                    if (win_end - win_start) > max_kmer_tokens:
                        win_end = win_start + max_kmer_tokens
                    if (win_end - win_start) > max_kmer_tokens:
                        win_start = win_end - max_kmer_tokens

                win_len = win_end - win_start
                if win_len <= 0:
                    continue
                if win_len > max_kmer_tokens:
                    win_end = win_start + max_kmer_tokens
                    win_len = win_end - win_start

                kmers_window = kmers[win_start:win_end]

                mut_token_ids_local = [
                    i_tok - win_start
                    for i_tok in mut_token_ids_full
                    if (win_start <= i_tok < win_end)
                ]
                if len(mut_token_ids_local) == 0:
                    continue

                kmer_str = " ".join(kmers_window)
                out = self.tokenizer(
                    kmer_str,
                    return_tensors="pt",
                )
                input_ids = out["input_ids"].to(self.device)

                outputs = self.model(
                    input_ids,
                    output_hidden_states=True,
                    return_dict=True,
                )
                last_hidden = outputs.hidden_states[-1].squeeze(0)

                selected_indices = [t + 1 for t in mut_token_ids_local]
                selected_embs = last_hidden[selected_indices, :]

                selected_raw = selected_embs.detach().cpu().tolist()
                rows.append([sample, varpos, selected_raw])

        return pd.DataFrame(rows, columns=["vcf_iid", "variant", "raw_token_embs"])

/home/tech/anaconda3/envs/benchmark/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from pathlib import Path

mutpool = DNABERT_MutPool(
    model_path="zhihan1996/DNA_bert_6",
    kmer=6,
    device="cuda:0",
)

bp = 50
seq_col = f"var_seq_{bp}bp"
idx_col = f"mut_idx_{bp}bp"

df_input = dnv[["vcf_iid", "variant", seq_col, idx_col]].copy()

df_raw = mutpool.get_mut_emb_raw(
    seq_df=df_input,
    seq_col=seq_col,
    mut_idx_col=idx_col,
)

/home/tech/anaconda3/envs/benchmark/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
100%|██████████| 5/5 [00:00<00:00,  5.86it/s]


In [5]:
import numpy as np

def add_poolings(df_raw):

    df_raw["mut_emb_max"] = [
        np.max(embs, axis=0).tolist()
        for embs in df_raw["raw_token_embs"]
    ]

    return df_raw


df_pooled = add_poolings(df_raw)

## 2. DNABERT-2

In [7]:
import torch
import pandas as pd
from transformers import AutoModel, AutoTokenizer, BertConfig
from tqdm import tqdm

class DNABERT2_MutPool:
    def __init__(self, model_path, device="cuda:1"):
        self.device = torch.device(device)
        self.config = BertConfig.from_pretrained(model_path)
        self.config.attention_probs_dropout_prob = 1e-6

        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(model_path, trust_remote_code=True, config=self.config)
        self.model.to(self.device)
        self.model.eval()

    def get_mut_emb_raw(
        self,
        seq_df: pd.DataFrame,
        seq_col: str,
        mut_idx_col: str,
    ):

        rows = []
        with torch.no_grad():
            for _, row in tqdm(seq_df.iterrows(), total=len(seq_df)):

                sample = row["vcf_iid"]
                varpos = row["variant"]
                seq = row[seq_col]
                mut_idx_list = row[mut_idx_col]

                if mut_idx_list is None or len(mut_idx_list) == 0:
                    continue

                out = self.tokenizer(
                    seq,
                    return_tensors="pt",
                    return_offsets_mapping=True
                )
                token_embs = self.model(out["input_ids"].to(self.device))[0].squeeze(0)
                offsets = out["offset_mapping"].squeeze(0).tolist()

                mut_start = min(mut_idx_list)
                mut_end   = max(mut_idx_list) + 1

                mut_token_ids = []
                for tok_idx, (s, e) in enumerate(offsets):
                    if (mut_start < e) and (mut_end > s):
                        mut_token_ids.append(tok_idx)

                selected_raw = token_embs[mut_token_ids, :].detach().cpu().tolist()

                rows.append([sample, varpos, selected_raw])

        return pd.DataFrame(rows, columns=["vcf_iid", "variant", "raw_token_embs"])

In [9]:
mutpool = DNABERT2_MutPool("zhihan1996/DNABERT-2-117M", device="cuda:0")

bp = 50
seq_col = f"var_seq_{bp}bp"
idx_col = f"mut_idx_{bp}bp"

df_raw = mutpool.get_mut_emb_raw(
    seq_df = dnv[["vcf_iid", "variant", seq_col, idx_col]],
    seq_col = seq_col,
    mut_idx_col = idx_col,
)

/home/tech/anaconda3/envs/benchmark/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 5/5 [00:00<00:00, 38.96it/s]


In [10]:
import numpy as np

def add_poolings(df_raw):

    df_raw["mut_emb_max"] = [
        np.max(embs, axis=0).tolist()
        for embs in df_raw["raw_token_embs"]
    ]

    return df_raw


df_pooled = add_poolings(df_raw)

## 3. Nucleotide Transformer V2

In [11]:
import gc
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForMaskedLM, AutoTokenizer


class NucleotideTransformerMutPool:

    def __init__(
        self,
        model_path: str = "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species",
        revision: str | None = None,
        device: str = "cuda:0",
        max_seq_len: int | None = None,
        max_tokens: int | None = None,
        use_amp: bool = True,
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            revision=revision,
            trust_remote_code=True,
        )
        self.model = AutoModelForMaskedLM.from_pretrained(
            model_path,
            revision=revision,
            trust_remote_code=True,
        ).to(self.device).eval()

        self.dim = self.model.config.hidden_size

        if max_seq_len is None:
            self.max_seq_len = int(self.tokenizer.model_max_length)
        else:
            self.max_seq_len = int(max_seq_len)

        if max_tokens is None:
            self.max_tokens = getattr(self.model.config, "max_position_embeddings", 2048)
        else:
            self.max_tokens = int(max_tokens)

        self.use_amp = bool(use_amp and self.device.type == "cuda")

    @staticmethod
    def _repeat_embedding_vectors(tokens, embeddings, has_special_tokens: bool = True):
        assert embeddings.ndim == 3
        assert len(tokens) == embeddings.shape[1]

        new_embeddings = []
        for idx, token in enumerate(tokens):
            if has_special_tokens and idx == 0:
                new_embeddings.append(embeddings[:, [idx]])
                continue

            token_embedding = embeddings[:, [idx]]
            new_embeddings.extend([token_embedding] * len(token))

        new_embeddings = np.concatenate(new_embeddings, axis=1)
        return new_embeddings

    def _embed_batch_per_base(self, seq_list: list[str]) -> list[np.ndarray]:

        seq_list_trunc = [s[: self.max_seq_len] for s in seq_list]

        autocast_ctx = (
            torch.autocast(device_type="cuda", dtype=torch.float16)
            if self.use_amp else torch.no_grad()
        )

        with torch.inference_mode(), autocast_ctx:
            encoded = self.tokenizer(
                seq_list_trunc,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=self.max_seq_len,
            )
            input_ids = encoded["input_ids"].to(self.device)
            B, L_tok = input_ids.shape

            if L_tok > self.max_tokens:
                outs_chunks = []
                for start in range(0, L_tok, self.max_tokens):
                    end = min(start + self.max_tokens, L_tok)
                    ids_chunk = input_ids[:, start:end]
                    out = self.model(
                        ids_chunk,
                        output_hidden_states=True,
                    )["hidden_states"][-1]
                    outs_chunks.append(out.detach().cpu())
                outs = torch.cat(outs_chunks, dim=1)
            else:
                out = self.model(
                    input_ids,
                    output_hidden_states=True,
                )["hidden_states"][-1]
                outs = out.detach().cpu()

        outs = outs.numpy()

        per_base_embs: list[np.ndarray] = []
        for b in range(B):
            ids_b = input_ids[b].cpu()
            tokens_b = self.tokenizer.convert_ids_to_tokens(ids_b)
            emb_b = outs[b:b+1, :, :]
            emb_b = self._repeat_embedding_vectors(tokens_b, emb_b)
            emb_b = emb_b[:, 1:, :]
            per_base_embs.append(emb_b[0])

        return per_base_embs

    def get_mut_emb_raw_from_df(
        self,
        seq_df: pd.DataFrame,
        seq_col: str,
        mut_idx_col: str,
        batch_size: int = 32,
        disable_tqdm: bool = False,
    ) -> pd.DataFrame:

        assert all(c in seq_df.columns for c in ["SAMPLE", "variant", seq_col, mut_idx_col])

        rows = []
        n = len(seq_df)

        for start in tqdm(range(0, n, batch_size), disable=disable_tqdm):
            end = min(start + batch_size, n)
            sub = seq_df.iloc[start:end].reset_index(drop=True)

            samples = sub["SAMPLE"].tolist()
            variants = sub["variant"].tolist()
            seqs = sub[seq_col].astype(str).tolist()
            mut_lists = sub[mut_idx_col].tolist()

            valid_idx = [i for i, ml in enumerate(mut_lists) if ml is not None and len(ml) > 0]
            if not valid_idx:
                continue

            seqs_batch = [seqs[i] for i in valid_idx]
            mut_lists_batch = [mut_lists[i] for i in valid_idx]
            samples_batch = [samples[i] for i in valid_idx]
            variants_batch = [variants[i] for i in valid_idx]

            per_base_embs = self._embed_batch_per_base(seqs_batch)

            for b, (sample, varpos, seq_emb, mut_idx_list) in enumerate(
                zip(samples_batch, variants_batch, per_base_embs, mut_lists_batch)
            ):
                L_bp, H = seq_emb.shape
                emb_indices = []
                for mi in mut_idx_list:
                    mi_int = int(mi)
                    if 0 <= mi_int < L_bp:
                        emb_indices.append(mi_int)

                if not emb_indices:
                    continue

                emb_indices = sorted(set(emb_indices))
                selected = seq_emb[emb_indices, :]

                rows.append([sample, varpos, selected.tolist()])

            gc.collect()
            if self.device.type == "cuda":
                torch.cuda.empty_cache()

        return pd.DataFrame(rows, columns=["SAMPLE", "variant", "raw_token_embs"])

In [12]:
offset = 50
seq_col = f"var_seq_{offset}bp"
idx_col = f"mut_idx_{offset}bp"

dnv_input = dnv[["vcf_iid", "variant", seq_col, idx_col]].copy()
dnv_input.columns = ["SAMPLE", "variant", seq_col, idx_col]

nt_mutpool = NucleotideTransformerMutPool(
    model_path="InstaDeepAI/nucleotide-transformer-v2-500m-multi-species",
    revision=None,
    device="cuda:1",
    max_seq_len=12000,
    max_tokens=2048,
    use_amp=True,
)

df_raw = nt_mutpool.get_mut_emb_raw_from_df(
    seq_df=dnv_input,
    seq_col=seq_col,
    mut_idx_col=idx_col,
    batch_size=512,
)

100%|██████████| 1/1 [00:00<00:00,  1.43it/s]


In [13]:
import numpy as np
import pandas as pd

def add_poolings_nt2(df_raw):

    maxs = []

    for e in df_raw["raw_token_embs"].to_numpy():
        arr = np.asarray(e)

        if arr.ndim == 1:
            if arr.dtype == object:
                mat = np.stack(arr, axis=0)
            else:
                mat = arr.reshape(1, -1)

        elif arr.ndim == 2:
            mat = arr

        elif arr.ndim == 3 and arr.shape[1] == 1:
            mat = arr[:, 0, :]

        else:
            raise ValueError(f"Unexpected raw_token_embs shape: {arr.shape}")

        mat = mat.astype(float, copy=False)

        maxs.append(mat.max(axis=0))

    df_raw = df_raw.copy()
    df_raw["mut_emb_max"] = [m.tolist() for m in maxs]

    return df_raw

df_pooled = add_poolings_nt2(df_raw)

## 4. Nulcleotide Transformer V3

In [7]:
import os
import gc
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoConfig, AutoTokenizer, AutoModel, AutoModelForMaskedLM

def resolve_device(device_str: str):
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    if device_str == "cuda":
        return torch.device("cuda:0")
    if device_str.startswith("cuda:"):
        idx = int(device_str.split(":")[1])
        if idx >= n:
            print(f"[WARN] requested {device_str} but only {n} visible GPUs. Fallback to cuda:0")
            return torch.device("cuda:0")
        return torch.device(device_str)
    return torch.device("cuda:0")


def infer_hidden_dim_from_config(cfg) -> int | None:
    candidates = ["hidden_size", "d_model", "dim", "model_dim", "embedding_dim", "n_embd"]
    for k in candidates:
        if hasattr(cfg, k):
            v = getattr(cfg, k)
            if isinstance(v, int) and v > 0:
                return v
    return None


class NTV3MutPool:
    def __init__(
        self,
        model_path: str,
        model_kind: str = "pre",
        revision: str | None = None,
        device: str = "cuda",
        use_amp: bool = True,
        pad_to_multiple_of: int = 128,
        hf_token: str | None = None,
    ):
        assert model_kind in ["pre", "post"]
        self.model_kind = model_kind
        self.device = resolve_device(device)

        if hf_token is None:
            hf_token = os.getenv("HF_TOKEN", None)

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            revision=revision,
            trust_remote_code=True,
            token=hf_token,
        )

        if self.model_kind == "pre":
            self.model = AutoModelForMaskedLM.from_pretrained(
                model_path,
                revision=revision,
                trust_remote_code=True,
                token=hf_token,
            )
        else:
            self.model = AutoModel.from_pretrained(
                model_path,
                revision=revision,
                trust_remote_code=True,
                token=hf_token,
            )

        self.model = self.model.to(self.device).eval()

        self.use_amp = bool(use_amp and self.device.type == "cuda")
        if self.device.type == "cuda":
            major, _ = torch.cuda.get_device_capability(0)
            self.amp_dtype = torch.bfloat16 if major >= 8 else torch.float16
        else:
            self.amp_dtype = torch.float32

        self.pad_to_multiple_of = pad_to_multiple_of

        if self.model_kind == "post" and not hasattr(self.model, "encode_species"):
            raise ValueError("post model requires model.encode_species(species_list)")

        self.dim = infer_hidden_dim_from_config(self.model.config)
        if self.dim is None:
            self.dim = self._infer_dim_by_dummy_forward()
        print(f"[NTv3] kind={self.model_kind} | hidden dim = {self.dim} | device={self.device}")

    @torch.no_grad()
    def _infer_dim_by_dummy_forward(self) -> int:
        dummy = "ACGT" * 8
        enc = self.tokenizer(
            [dummy],
            add_special_tokens=False,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        enc = {k: v.to(self.device) for k, v in enc.items()}

        species_ids = None
        if self.model_kind == "post":
            species_ids = self.model.encode_species(["human"])
            if isinstance(species_ids, np.ndarray):
                species_ids = torch.from_numpy(species_ids)
            elif not torch.is_tensor(species_ids):
                species_ids = torch.tensor(species_ids)
            species_ids = species_ids.to(self.device)

        autocast_ctx = (
            torch.autocast(device_type="cuda", dtype=self.amp_dtype)
            if self.use_amp else torch.no_grad()
        )

        with torch.inference_mode(), autocast_ctx:
            if species_ids is None:
                out = self.model(**enc, output_hidden_states=True, return_dict=True)
            else:
                out = self.model(
                    input_ids=enc["input_ids"],
                    species_ids=species_ids,
                    output_hidden_states=True,
                    return_dict=True
                )

        h = out["hidden_states"][-1]
        return int(h.shape[-1])

    @torch.no_grad()
    def _forward_last_hidden(self, encoded, species_list=None):
        encoded = {k: v.to(self.device) for k, v in encoded.items()}

        species_ids = None
        if self.model_kind == "post":
            if species_list is None:
                species_list = ["human"] * encoded["input_ids"].shape[0]
            species_ids = self.model.encode_species(species_list)
            if isinstance(species_ids, np.ndarray):
                species_ids = torch.from_numpy(species_ids)
            elif not torch.is_tensor(species_ids):
                species_ids = torch.tensor(species_ids)
            species_ids = species_ids.to(self.device)

        autocast_ctx = (
            torch.autocast(device_type="cuda", dtype=self.amp_dtype)
            if self.use_amp else torch.no_grad()
        )

        with torch.inference_mode(), autocast_ctx:
            if species_ids is None:
                out = self.model(**encoded, output_hidden_states=True, return_dict=True)
            else:
                out = self.model(
                    input_ids=encoded["input_ids"],
                    species_ids=species_ids,
                    output_hidden_states=True,
                    return_dict=True
                )

        hidden = out["hidden_states"][-1].detach().cpu()
        attn_mask = encoded.get("attention_mask", torch.ones_like(encoded["input_ids"])).detach().cpu()
        return hidden, attn_mask

    def _embed_batch_token_level(self, seq_list, species_list=None):
        encoded = self.tokenizer(
            seq_list,
            add_special_tokens=False,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        return self._forward_last_hidden(encoded, species_list=species_list)

    def _validate_no_special_tokens(self, seq: str, show_details: bool = True):
        enc = self.tokenizer(
            [seq],
            add_special_tokens=False,
            padding=False,
            return_tensors="pt",
        )
        
        input_ids = enc["input_ids"][0]
        tokens = self.tokenizer.convert_ids_to_tokens(input_ids)
        
        bp_len = len(seq)
        token_len = len(tokens)
        
        if bp_len != token_len:
            msg = f"bp_len({bp_len}) != token_len({token_len})"
            return False, msg
        
        special_ids = set(self.tokenizer.all_special_ids)
        has_special = any(tid.item() in special_ids for tid in input_ids)
        
        if has_special:
            special_tokens_found = [
                (i, self.tokenizer.convert_ids_to_tokens([tid.item()])[0]) 
                for i, tid in enumerate(input_ids) 
                if tid.item() in special_ids
            ]
            msg = f"Special tokens found: {special_tokens_found}"
            return False, msg
        
        first_token = tokens[0]
        last_token = tokens[-1]
        
        suspicious = ['[CLS]', '[SEP]', '<s>', '</s>', '<cls>', '<sep>', '<CLS>', '<SEP>']
        if first_token in suspicious or last_token in suspicious:
            msg = f"Suspicious tokens: first='{first_token}', last='{last_token}'"
            return False, msg
        
        if show_details:
            print(f"Validation passed:")
            print(f"bp_len={bp_len}, token_len={token_len}")
            print(f"first_token='{first_token}', last_token='{last_token}'")
            print(f"first_5_bp: {seq[:5]}")
            print(f"first_5_tok: {''.join(tokens[:5])}")
        
        return True, "OK"

    def debug_show_mut_tokens(
        self,
        seq: str,
        mut_idx_list,
        species: str = "human",
        window: int = 12,
        show_tokens: bool = True,
        validate: bool = True,
    ):
        if validate:
            print(f"\n{'='*60}")
            is_valid, msg = self._validate_no_special_tokens(seq, show_details=True)
            print(f"{'='*60}")
            if not is_valid:
                print(f"\n{msg}\n")
        
        mut_idx_list = sorted({int(x) for x in mut_idx_list})

        enc = self.tokenizer(
            [seq],
            add_special_tokens=False,
            padding=False,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"][0]
        tokens = self.tokenizer.convert_ids_to_tokens(input_ids)

        print(f"\nbp_len={len(seq)} | token_len={len(tokens)}")
        if len(seq) != len(tokens):
            print("bp_len != token_len")

        for mi in mut_idx_list[:10]:
            lo = max(0, mi - window)
            hi = min(len(seq), mi + window + 1)

            seq_slice = seq[lo:hi]
            caret = " " * (mi - lo) + "^"

            print("\n---")
            print(f"mut_idx={mi}")
            print(seq_slice)
            print(caret)

            if show_tokens:
                if mi < len(tokens):
                    t_lo = lo
                    t_hi = hi
                    tok_slice = tokens[t_lo:t_hi]
                    print("TOK:", "".join(tok_slice))
                    print("     " + caret)

    def get_mut_emb_raw_from_df(
        self,
        seq_df: pd.DataFrame,
        seq_col: str,
        mut_idx_col: str,
        batch_size: int = 64,
        species_col: str | None = None,
        disable_tqdm: bool = False,
        validate_first_batch: bool = True,
    ) -> pd.DataFrame:

        assert all(c in seq_df.columns for c in ["SAMPLE", "variant", seq_col, mut_idx_col])

        rows = []
        n = len(seq_df)
        first_batch = True

        for start in tqdm(range(0, n, batch_size), disable=disable_tqdm):
            end = min(start + batch_size, n)
            sub = seq_df.iloc[start:end].reset_index(drop=True)

            samples = sub["SAMPLE"].tolist()
            variants = sub["variant"].tolist()
            seqs = sub[seq_col].astype(str).tolist()
            mut_lists = sub[mut_idx_col].tolist()

            if first_batch and validate_first_batch:
                print(f"\n{'='*60}")
                print(f"Checking first sequence for special tokens...")
                print(f"{'='*60}")
                
                first_seq = seqs[0]
                is_valid, msg = self._validate_no_special_tokens(first_seq, show_details=True)
                
                if not is_valid:
                    print(f"\nWARNING: {msg}")
                    user_input = input("\nContinue anyway? (y/n): ")
                    if user_input.lower() != 'y':
                        raise ValueError("Validation failed. Stopping.")
                
                print(f"{'='*60}\n")
                first_batch = False

            if self.model_kind == "post":
                if species_col is not None and species_col in sub.columns:
                    species_list = sub[species_col].astype(str).tolist()
                else:
                    species_list = ["human"] * len(sub)
            else:
                species_list = None

            valid_idx = [i for i, ml in enumerate(mut_lists) if ml is not None and len(ml) > 0]
            if not valid_idx:
                continue

            seqs_batch = [seqs[i] for i in valid_idx]
            mut_lists_batch = [mut_lists[i] for i in valid_idx]
            samples_batch = [samples[i] for i in valid_idx]
            variants_batch = [variants[i] for i in valid_idx]
            species_batch = [species_list[i] for i in valid_idx] if species_list is not None else None

            hidden, attn_mask = self._embed_batch_token_level(seqs_batch, species_list=species_batch)
            B, T, H = hidden.shape

            for b in range(B):
                L = int(attn_mask[b].sum().item())

                emb_indices = []
                for mi in mut_lists_batch[b]:
                    mi = int(mi)
                    if 0 <= mi < L:
                        emb_indices.append(mi)

                if not emb_indices:
                    continue

                emb_indices = sorted(set(emb_indices))
                selected = hidden[b, emb_indices, :]
                rows.append([samples_batch[b], variants_batch[b], selected.numpy().tolist()])

            gc.collect()
            if self.device.type == "cuda":
                torch.cuda.empty_cache()

        return pd.DataFrame(rows, columns=["SAMPLE", "variant", "raw_token_embs"])

ntv3_post = NTV3MutPool(
    model_path="InstaDeepAI/NTv3_650M_post",
    model_kind="post",
    device="cuda",
    use_amp=True
)

[NTv3] kind=post | hidden dim = 1536 | device=cuda:0


In [9]:
offset=50

seq_col = f"var_seq_{offset}bp"
idx_col = f"mut_idx_{offset}bp"

dnv_input = dnv[["vcf_iid", "variant", seq_col, idx_col]].copy()
dnv_input.columns = ["SAMPLE", "variant", seq_col, idx_col]

In [10]:
df_raw_post = ntv3_post.get_mut_emb_raw_from_df(
    seq_df=dnv_input,
    seq_col=seq_col,
    mut_idx_col=idx_col,
    batch_size=1500,
    validate_first_batch=True
)

  0%|          | 0/1 [00:00<?, ?it/s]


Checking first sequence for special tokens...
Validation passed:
bp_len=101, token_len=101
first_token='T', last_token='G'
first_5_bp: TTATG
first_5_tok: TTATG



100%|██████████| 1/1 [00:00<00:00,  3.46it/s]


In [11]:
def add_poolings_ntv3(df_raw):
    maxs  = []
    means = []

    for e in df_raw["raw_token_embs"].to_numpy():

        arr = np.asarray(e)

        if arr.ndim == 1:
            mat = arr.reshape(1, -1)

        elif arr.ndim == 2:
            mat = arr

        elif arr.ndim == 3 and arr.shape[1] == 1:
            mat = arr[:, 0, :]

        else:
            raise ValueError(f"Unexpected shape: {arr.shape}")

        mat = mat.astype(float, copy=False)

        maxs.append(mat.max(axis=0))
        means.append(mat.mean(axis=0))

    df_out = df_raw.copy()
    df_out["mut_emb_max"]  = [m.tolist() for m in maxs]
    df_out["mut_emb_mean"] = [m.tolist() for m in means]

    return df_out

df_pooled_post = add_poolings_ntv3(df_raw_post)

## 5. HyenaDNA

## 모델 코드

In [14]:
import os
import gc
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np


class HyenaDNA:

    def __init__(
        self,
        ckpt_path: str,
        length: int,
        pad_idx: int = 4,
        device: str = "cuda:0",
        batch_size: int = 128,
    ):
        self.tokenizer = AutoTokenizer.from_pretrained(
            ckpt_path,
            trust_remote_code=True
        )

        self.model = AutoModel.from_pretrained(
            ckpt_path,
            trust_remote_code=True
        )

        self.pad_idx = pad_idx
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.dim = int(self.model.config.d_model)
        self.length = int(length)
        self.batch_size = int(batch_size)

        self.model.to(self.device)
        self.model.eval()

    def _ensure_list(self, x):

        if isinstance(x, (list, tuple, np.ndarray)):
            return list(x)

        if pd.isna(x):
            return []

        if isinstance(x, str):
            import ast
            try:
                v = ast.literal_eval(x)
                if isinstance(v, (list, tuple, np.ndarray)):
                    return list(v)
                else:
                    return [int(v)]
            except Exception:
                try:
                    return [int(p) for p in x.split(",") if p != ""]
                except Exception:
                    return []

        try:
            return [int(x)]
        except Exception:
            return []

    def get_embeddings_from_df(
        self,
        seq_df: pd.DataFrame,
        seq_col: str,
        len_col: str,
        idx_col: str,
        debug: bool = False,
        debug_n: int = 3,
    ) -> pd.DataFrame:

        required = ["vcf_iid", "variant", seq_col, len_col, idx_col]
        missing = [c for c in required if c not in seq_df.columns]

        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        out_rows = []
        debug_count = 0

        with torch.inference_mode():

            n = len(seq_df)

            for start in tqdm(range(0, n, self.batch_size)):

                end = min(start + self.batch_size, n)
                batch = seq_df.iloc[start:end]

                seqs_fwd = batch[seq_col].tolist()
                seqs_rev = [s[::-1] for s in seqs_fwd]

                lens = batch[len_col].astype(int).tolist()
                mut_arrs = batch[idx_col].tolist()

                toks_fwd = self.tokenizer(
                    seqs_fwd,
                    add_special_tokens=False,
                    padding="max_length",
                    truncation=True,
                    max_length=self.length,
                    return_tensors="pt",
                )

                toks_rev = self.tokenizer(
                    seqs_rev,
                    add_special_tokens=False,
                    padding="max_length",
                    truncation=True,
                    max_length=self.length,
                    return_tensors="pt",
                )

                input_ids_fwd = toks_fwd["input_ids"].to(self.device)
                input_ids_rev = toks_rev["input_ids"].to(self.device)

                mask_fwd = (input_ids_fwd != self.pad_idx)
                mask_rev = (input_ids_rev != self.pad_idx)

                token_embs_fwd = self.model(input_ids_fwd)[0]
                token_embs_rev = self.model(input_ids_rev)[0]

                B, L, D = token_embs_fwd.shape

                token_embs_fwd = token_embs_fwd * mask_fwd.unsqueeze(-1)
                token_embs_rev = token_embs_rev * mask_rev.unsqueeze(-1)

                for i in range(B):

                    vcf_iid = batch["vcf_iid"].iloc[i]
                    variant = batch["variant"].iloc[i]
                    seq_len = int(lens[i])
                    seq_fwd = seqs_fwd[i]

                    valid_pos_fwd = torch.nonzero(mask_fwd[i], as_tuple=True)[0]
                    valid_pos_rev = torch.nonzero(mask_rev[i], as_tuple=True)[0]

                    seq_len_eff = int(valid_pos_fwd.numel())

                    if seq_len_eff != seq_len:
                        seq_len = seq_len_eff

                    arr = np.array(mut_arrs[i])
                    mut_idx = [int(x) for x in arr.tolist() if 0 <= int(x) < seq_len]

                    if mut_idx:

                        nuc_idx_fwd = torch.tensor(
                            mut_idx,
                            dtype=torch.long,
                            device=self.device
                        )

                        token_idx_fwd = valid_pos_fwd[nuc_idx_fwd]
                        mut_embs_fwd_t = token_embs_fwd[i, token_idx_fwd]

                        mut_emb_max_fwd_t = mut_embs_fwd_t.max(dim=0).values

                        rev_nuc_idx = [seq_len - 1 - j for j in mut_idx]

                        nuc_idx_rev = torch.tensor(
                            rev_nuc_idx,
                            dtype=torch.long,
                            device=self.device
                        )

                        token_idx_rev = valid_pos_rev[nuc_idx_rev]
                        mut_embs_rev_t = token_embs_rev[i, token_idx_rev]

                        mut_emb_max_rev_t = mut_embs_rev_t.max(dim=0).values

                        mut_emb_fwd_rev_concat_t = torch.cat(
                            [mut_emb_max_fwd_t, mut_emb_max_rev_t],
                            dim=0
                        )

                    else:

                        mut_emb_fwd_rev_concat_t = torch.full(
                            (self.dim * 2,),
                            float("nan"),
                            device=self.device,
                            dtype=token_embs_fwd.dtype,
                        )

                    out_rows.append(
                        {
                            "vcf_iid": vcf_iid,
                            "variant": variant,
                            "mut_emb_fwd_rev_concat": mut_emb_fwd_rev_concat_t
                            .detach()
                            .cpu()
                            .numpy(),
                        }
                    )

                del batch, seqs_fwd, seqs_rev, lens, mut_arrs
                del toks_fwd, toks_rev
                del input_ids_fwd, input_ids_rev
                del mask_fwd, mask_rev
                del token_embs_fwd, token_embs_rev

                gc.collect()

                if self.device.type == "cuda":
                    torch.cuda.empty_cache()

        out_df = pd.DataFrame(out_rows)

        return out_df

In [15]:
import os
import pandas as pd

bp = 50

seq_col = f"var_seq_{bp}bp"
len_col = f"var_len_{bp}bp"
idx_col = f"mut_idx_{bp}bp"

max_len = int(dnv[len_col].max())

print("max_len:", max_len)

hyenadna_embedder = HyenaDNA(
    ckpt_path="LongSafari/hyenadna-large-1m-seqlen-hf",
    length=max_len,
    pad_idx=4,
    device="cuda:1",
    batch_size=512,
)

out_df = hyenadna_embedder.get_embeddings_from_df(
    seq_df=dnv,
    seq_col=seq_col,
    len_col=len_col,
    idx_col=idx_col,
)

max_len: 101


100%|██████████| 1/1 [00:00<00:00,  6.96it/s]


## 6. Evo 2

In [2]:
import gc
import warnings
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from evo2 import Evo2

warnings.filterwarnings("ignore")


class Evo2MutationEmbedder:

    def __init__(
        self,
        model_name="evo2_7b",
        layer_name="blocks.26.mlp.l3",
        device="cuda:0",
        batch_size=64,
    ):

        self.model_name = model_name
        self.layer_name = layer_name
        self.batch_size = batch_size

        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        print("Loading Evo2 model...")
        self.model = Evo2(model_name)
        self.tokenizer = self.model.tokenizer

        for p in self.model.model.parameters():
            p.data = p.data.to(self.device)

        for b in self.model.model.buffers():
            b.data = b.data.to(self.device)

        self.model.model.to(self.device)

        if torch.cuda.is_available():
            self.model.model.to(dtype=torch.bfloat16)

        if hasattr(self.model.model, "block_idx_to_device"):
            self.model.model.block_idx_to_device = {
                k: self.device for k in self.model.model.block_idx_to_device
            }

        self.model.model.eval()

    def _ensure_list(self, x):

        if isinstance(x, (list, tuple, np.ndarray)):
            return [int(v) for v in np.array(x).tolist()]

        if pd.isna(x):
            return []

        if isinstance(x, str):
            try:
                import ast
                v = ast.literal_eval(x)
                if isinstance(v, (list, tuple, np.ndarray)):
                    return [int(t) for t in list(v)]
                return [int(v)]
            except Exception:
                try:
                    return [int(p) for p in x.split(",") if p != ""]
                except Exception:
                    return []

        try:
            return [int(x)]
        except Exception:
            return []

    @torch.inference_mode()
    def get_embeddings_from_df(
        self,
        seq_df: pd.DataFrame,
        seq_col: str,
        len_col: str,
        idx_col: str,
    ):

        seqs = seq_df[seq_col].astype(str).str.upper().str.replace("U", "T", regex=False).tolist()
        lens = seq_df[len_col].astype(int).tolist()
        mut_idxs = [self._ensure_list(x) for x in seq_df[idx_col].tolist()]

        toks = [self.tokenizer.tokenize(s) for s in seqs]
        tok_lens = [len(t) for t in toks]

        buckets = {}
        for i, L in enumerate(tok_lens):
            buckets.setdefault(L, []).append(i)

        out_emb = [None] * len(seq_df)

        for L, idxs in buckets.items():

            for s in tqdm(range(0, len(idxs), self.batch_size)):

                bi = idxs[s:s + self.batch_size]

                input_ids_fwd = torch.tensor(
                    [toks[i] for i in bi],
                    dtype=torch.long,
                    device=self.device,
                )

                input_ids_rev = torch.flip(input_ids_fwd, dims=[1])

                if torch.cuda.is_available():
                    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):

                        _, emb_fwd = self.model(
                            input_ids_fwd,
                            return_embeddings=True,
                            layer_names=[self.layer_name],
                        )

                        _, emb_rev = self.model(
                            input_ids_rev,
                            return_embeddings=True,
                            layer_names=[self.layer_name],
                        )

                else:
                    _, emb_fwd = self.model(
                        input_ids_fwd,
                        return_embeddings=True,
                        layer_names=[self.layer_name],
                    )

                    _, emb_rev = self.model(
                        input_ids_rev,
                        return_embeddings=True,
                        layer_names=[self.layer_name],
                    )

                h_fwd = emb_fwd[self.layer_name]
                h_rev = emb_rev[self.layer_name]

                B, L, D = h_fwd.shape

                mut_mask = torch.zeros((B, L), dtype=torch.bool, device=self.device)
                rev_mask = torch.zeros((B, L), dtype=torch.bool, device=self.device)

                for bpos, ridx in enumerate(bi):

                    mids = [m for m in mut_idxs[ridx] if 0 <= m < L]

                    if len(mids) == 0:
                        continue

                    mut_mask[bpos, mids] = True

                    rev_ids = [(L - 1 - m) for m in mids]
                    rev_mask[bpos, rev_ids] = True

                neg_inf = torch.tensor(-1e9, device=self.device, dtype=h_fwd.dtype)

                fwd_sel = h_fwd.masked_fill(
                    ~mut_mask.unsqueeze(-1),
                    neg_inf
                ).max(dim=1).values

                rev_sel = h_rev.masked_fill(
                    ~rev_mask.unsqueeze(-1),
                    neg_inf
                ).max(dim=1).values

                fwd_sel = torch.where(
                    torch.isclose(fwd_sel, neg_inf),
                    torch.tensor(float("nan"), device=self.device, dtype=fwd_sel.dtype),
                    fwd_sel
                )

                rev_sel = torch.where(
                    torch.isclose(rev_sel, neg_inf),
                    torch.tensor(float("nan"), device=self.device, dtype=rev_sel.dtype),
                    rev_sel
                )

                mut_concat = torch.cat([fwd_sel, rev_sel], dim=1)

                mut_concat = mut_concat.detach().float().cpu().numpy()

                for k, ridx in enumerate(bi):
                    out_emb[ridx] = mut_concat[k].astype(np.float32)

        out_df = seq_df[["vcf_iid", "variant"]].copy()
        out_df["mut_emb"] = out_emb

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return out_df

/home/tech/anaconda3/envs/evo2_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

bp = 50

seq_col = f"var_seq_{bp}bp"
len_col = f"var_len_{bp}bp"
idx_col = f"mut_idx_{bp}bp"

evo2_embedder = Evo2MutationEmbedder(
    model_name="evo2_7b",
    layer_name="blocks.26.mlp.l3",
    device="cuda:0",
    batch_size=64,
)

out_df = evo2_embedder.get_embeddings_from_df(
    seq_df=dnv,
    seq_col=seq_col,
    len_col=len_col,
    idx_col=idx_col,
)

Loading Evo2 model...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 66052.03it/s]

Found complete file in repo: evo2_7b.pt


[03/05/26 11:58:08] INFO     StripedHyena - INFO - Initializing StripedHyena with config:              ]8;id=981403;file:///home/tech/anaconda3/envs/evo2_env/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=254265;file:///home/tech/anaconda3/envs/evo2_env/lib/python3.12/site-packages/vortex/model/model.py#616\616]8;;\
                             {'model_name': 'shc-evo2-7b-8k-2T-v2', 'vocab_size': 512, 'hidden_size':              
                             4096, 'num_filters': 4096, 'hcl_layer_idxs': [2, 6, 9, 13, 16, 20, 23,                
                             27, 30], 'hcm_layer_idxs': [1, 5, 8, 12, 15, 19, 22, 26, 29],                         
                             'hcs_layer_idxs': [0, 4, 7, 11, 14, 18, 21, 25, 28], 'attn_layer_idxs':               
                             [3, 10, 17, 24, 31], 'hcm_filter_length': 128, 'hcl_filter_groups': 4096,             
                             'hcm_filter_groups': 256, 'hcs_filter_groups': 256, 'hcs_filter_length':              
                             7, 'num_layers': 32, 'short_filter_length': 3, 'num_attention_heads': 32,             
                             'short_filter_bias': False, 'mlp_init_method': 'torch.nn.init.zeros_',                
                             'mlp_output_init_method': 'torch.nn.init.zeros_', 'eps': 1e-06,                       
                             'state_size': 16, 'rotary_emb_base': 10000, 'rotary_emb_scaling_factor':              
                             128, 'use_interpolated_rotary_pos_emb': True,                                         
                             'make_vocab_size_divisible_by': 8, 'inner_size_multiple_of': 16,                      
                             'inner_mlp_size': 11264, 'log_intermediate_values': False, 'proj_groups':             
                             1, 'hyena_filter_groups': 1, 'column_split_hyena': False, 'column_split':             
                             True, 'interleave': True, 'evo2_style_activations': True,                             
                             'model_parallel_size': 1, 'pipe_parallel_size': 1, 'tie_embeddings':                  
                             True, 'mha_out_proj_bias': True, 'hyena_out_proj_bias': True,                         
                             'hyena_flip_x1x2': False, 'qkv_proj_bias': False,                                     
                             'use_fp8_input_projections': True, 'max_seqlen': 1048576,                             
                             'max_batch_size': 1, 'final_norm': True, 'use_flash_attn': True,                      
                             'use_flash_rmsnorm': False, 'use_flash_depthwise': False, 'use_flashfft':             
                             False, 'use_laughing_hyena': False, 'inference_mode': True,                           
                             'tokenizer_type': 'CharLevelTokenizer', 'prefill_style': 'fft',                       
                             'mlp_activation': 'gelu', 'print_activations': False, 'Loader': <class                
                             'yaml.loader.FullLoader'>}                                                            

                    INFO     StripedHyena - INFO - Initializing 32 blocks...                           ]8;id=874300;file:///home/tech/anaconda3/envs/evo2_env/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=100948;file:///home/tech/anaconda3/envs/evo2_env/lib/python3.12/site-packages/vortex/model/model.py#635\635]8;;\

...

                    INFO     StripedHyena - INFO - Initialized model                                   ]8;id=692621;file:///home/tech/anaconda3/envs/evo2_env/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=350179;file:///home/tech/anaconda3/envs/evo2_env/lib/python3.12/site-packages/vortex/model/model.py#680\680]8;;\

                    INFO     vortex.model.utils - INFO - Loading                                        ]8;id=996691;file:///home/tech/anaconda3/envs/evo2_env/lib/python3.12/site-packages/vortex/model/utils.py\utils.py]8;;\:]8;id=527221;file:///home/tech/anaconda3/envs/evo2_env/lib/python3.12/site-packages/vortex/model/utils.py#92\92]8;;\
                             /home/tech/.cache/huggingface/hub/models--arcinstitute--evo2_7b/snapshots/            
                             bda0089f92582d5baabf0f22d9fc85f3588f6b58/evo2_7b.pt                                   

Extra keys in state_dict: {'blocks.17.mixer.dense._extra_state', 'blocks.13.mixer.mixer.filter.t', 'blocks.24.mixer.dense._extra_state', 'blocks.31.mixer.dense._extra_state', 'blocks.3.mixer.dense._extra_state', 'blocks.9.mixer.mixer.filter.t', 'blocks.10.mixer.dense._extra_state', 'blocks.24.mixer.attn._extra_state', 'blocks.27.mixer.mixer.filter.t', 'blocks.17.mixer.attn._extra_state', 'blocks.20.mixer.mixer.filter.t', 'blocks.10.mixer.attn._extra_state', 'unembed.weight', 'blocks.30.mixer.mixer.filter.t', 'blocks.6.mixer.mixer.filter.t', 'blocks.2.mixer.mixer.filter.t', 'blocks.16.mixer.mixer.filter.t', 'blocks.23.mixer.mixer.filter.t', 'blocks.3.mixer.attn._extra_state', 'blocks.31.mixer.attn._extra_state'}


[03/05/26 11:58:09] INFO     StripedHyena - INFO - Adjusting Wqkv for column split (permuting rows)    ]8;id=305925;file:///home/tech/anaconda3/envs/evo2_env/lib/python3.12/site-packages/vortex/model/model.py\model.py]8;;\:]8;id=550756;file:///home/tech/anaconda3/envs/evo2_env/lib/python3.12/site-packages/vortex/model/model.py#964\964]8;;\

100%|██████████| 1/1 [00:00<00:00, 13.77it/s]


## 7. PhyloGPN

In [2]:
import os
import gc
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer


class PhyloGPN_MutPool:

    def __init__(
        self,
        ckpt_path: str = "songlab/PhyloGPN",
        revision: str | None = None,
        length: int = 2200,
        pad_token: str = None,
        device: str = "cuda:0",
        batch_size: int = 128,
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(
            ckpt_path,
            revision=revision,
            trust_remote_code=True,
        )
        self.model = AutoModel.from_pretrained(
            ckpt_path,
            revision=revision,
            trust_remote_code=True,
        ).to(self.device).eval()

        self.dim = getattr(self.model.config, "outer_dim", None) or getattr(
            self.model.config, "hidden_size", None
        )
        if self.dim is None:
            raise ValueError("PhyloGPN embedding dimension not found.")

        self.length = int(length)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token or self.tokenizer.sep_token
        self.pad_token = pad_token if pad_token is not None else self.tokenizer.pad_token

        self.pad_flank = 481 // 2
        self.batch_size = int(batch_size)

    def _pad_biside(self, seq: str) -> str:
        return (self.pad_token * self.pad_flank) + seq + (self.pad_token * self.pad_flank)

    def get_mut_emb_raw(
        self,
        seq_df: pd.DataFrame,
        seq_col: str,
        mut_idx_col: str,
    ) -> pd.DataFrame:

        assert all(c in seq_df.columns for c in ["SAMPLE", "variant", seq_col, mut_idx_col])

        rows = []
        dim = self.dim

        with torch.inference_mode():
            n = len(seq_df)
            for start in tqdm(range(0, n, self.batch_size)):
                end = min(start + self.batch_size, n)
                sub = seq_df.iloc[start:end].reset_index(drop=True)

                samples = sub["SAMPLE"].tolist()
                variants = sub["variant"].tolist()
                seqs_raw = sub[seq_col].astype(str).tolist()
                mut_lists = sub[mut_idx_col].tolist()

                valid_mask = [(ml is not None and len(ml) > 0) for ml in mut_lists]
                if not any(valid_mask):
                    continue

                seqs_padded = [self._pad_biside(s) for s in seqs_raw]

                toks = self.tokenizer(
                    seqs_padded,
                    add_special_tokens=False,
                    padding="max_length",
                    truncation=True,
                    max_length=self.length,
                    return_tensors="pt",
                )
                input_ids = toks["input_ids"].to(self.device)

                token_embs = self.model.get_embeddings(input_ids)
                B, L_emb, D = token_embs.shape
                assert D == dim

                for b in range(B):
                    if not valid_mask[b]:
                        continue

                    sample = samples[b]
                    varpos = variants[b]
                    seq_raw = seqs_raw[b]
                    mut_idx_list = mut_lists[b]

                    seq_len = len(seq_raw)

                    emb_indices = []
                    for mi in mut_idx_list:
                        mi_int = int(mi)
                        if 0 <= mi_int < seq_len:
                            emb_idx = mi_int
                            if 0 <= emb_idx < L_emb:
                                emb_indices.append(emb_idx)

                    if len(emb_indices) == 0:
                        continue

                    emb_indices = sorted(set(int(i) for i in emb_indices))

                    idx_tensor = torch.tensor(
                        emb_indices,
                        device=token_embs.device,
                        dtype=torch.long,
                    )

                    selected = token_embs[b].index_select(0, idx_tensor)
                    selected_raw = selected.detach().cpu().tolist()

                    rows.append([sample, varpos, selected_raw])

                del toks, input_ids, token_embs
                gc.collect()
                if self.device.type == "cuda":
                    torch.cuda.empty_cache()

        return pd.DataFrame(rows, columns=["SAMPLE", "variant", "raw_token_embs"])

In [5]:
offset = 50

seq_col = f"var_seq_{offset}bp"
idx_col = f"mut_idx_{offset}bp"

dnv_input = dnv[["vcf_iid", "variant", seq_col, idx_col]].copy()
dnv_input.columns = ["SAMPLE", "variant", seq_col, idx_col]

ckpt = "songlab/PhyloGPN"
rev = "main"

phylo_mutpool = PhyloGPN_MutPool(
    ckpt_path=ckpt,
    revision=rev,
    length=offset * 2 + 100,
    device="cuda:0",
    batch_size=2048,
)

maxlen = dnv[seq_col].str.len().max()
print("maxlen(raw seq):", maxlen)

phylo_mutpool.length = int(maxlen + 2 * phylo_mutpool.pad_flank)

df_raw = phylo_mutpool.get_mut_emb_raw(
    seq_df=dnv_input,
    seq_col=seq_col,
    mut_idx_col=idx_col,
)

maxlen(raw seq): 101


100%|██████████| 1/1 [00:00<00:00,  3.40it/s]


In [6]:
import numpy as np
import pandas as pd

def add_poolings_phylo(df_raw, raw_col="raw_token_embs"):

    mats = []

    for e in df_raw[raw_col]:
        arr = np.array(e, dtype=object)

        if arr.dtype == object:
            arr = np.vstack([np.array(x) for x in arr])

        if arr.ndim == 1:
            arr = arr.reshape(1, -1)

        if arr.ndim == 3 and arr.shape[1] == 1:
            arr = arr[:, 0, :]

        arr = arr.astype(float)
        mats.append(arr)

    df_raw["mut_emb_max"] = [m.max(axis=0).tolist() for m in mats]

    return df_raw

df_pooled = add_poolings_phylo(df_raw, raw_col="raw_token_embs")